# Data modeling and query optimization

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-11-modeling-and-optimization.ipynb)

Two decisions shape almost every analytics query: how the data is **modeled**, and how the engine **executes** it.
Good modeling makes the common question a simple join. Good optimization makes the join cheap. This chapter covers
the grain of a table, the star schema, slowly changing dimensions, reading a query plan, and filters the engine can
use. It ends with practice questions and answers.

## Setup

Install PySpark, start a Spark session, and create the example tables.

In [ ]:
!pip install -q pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW customers AS
SELECT * FROM VALUES
  (1, 'Acme Corp',    'North', DATE '2022-01-05'),
  (2, 'Beta LLC',     'North', DATE '2022-02-10'),
  (3, 'Cyan Inc',     'South', DATE '2022-03-15'),
  (4, 'Delta Co',     'South', DATE '2022-04-20'),
  (5, 'Echo Ltd',     'West',  DATE '2022-05-25'),
  (6, 'Foxtrot GmbH', 'West',  DATE '2022-06-30'),
  (7, 'Gamma SA',     'East',  DATE '2023-01-10'),
  (8, 'Hotel BV',     'East',  DATE '2023-02-15')
AS customers(customer_id, name, region, signup_date)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW orders AS
SELECT * FROM VALUES
  (1,  1, 'completed', 100, DATE '2023-01-05'),
  (2,  1, 'completed', 200, DATE '2023-02-10'),
  (3,  1, 'completed', 150, DATE '2023-03-15'),
  (4,  2, 'completed', 300, DATE '2023-01-20'),
  (5,  2, 'cancelled', 250, DATE '2023-02-05'),
  (6,  2, 'completed', 400, DATE '2023-03-20'),
  (7,  3, 'completed', 120, DATE '2023-01-25'),
  (8,  3, 'completed', 180, DATE '2023-02-28'),
  (9,  4, 'pending',   220, DATE '2023-02-14'),
  (10, 4, 'completed', 320, DATE '2023-03-30'),
  (11, 5, 'completed', 500, DATE '2023-01-02'),
  (12, 5, 'completed', 450, DATE '2023-03-05'),
  (13, 6, 'completed', 275, DATE '2023-02-18'),
  (14, 6, 'cancelled', 325, DATE '2023-03-25')
AS orders(order_id, customer_id, status, amount, order_date)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW segment_history AS
SELECT * FROM VALUES
  (1, 'standard', TIMESTAMP '2026-01-01 00:00:00', TIMESTAMP '2026-06-01 00:00:00'),
  (1, 'premium',  TIMESTAMP '2026-06-01 00:00:00', NULL),
  (2, 'standard', TIMESTAMP '2026-01-01 00:00:00', NULL)
AS segment_history(customer_id, segment, effective_from, effective_to)
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW txns AS
SELECT * FROM VALUES
  (1, 1,  TIMESTAMP '2026-01-15 12:00:00', 20.0),
  (2, 1,  TIMESTAMP '2026-07-01 12:00:00', 30.0),
  (3, 2,  TIMESTAMP '2026-03-01 12:00:00', 40.0),
  (4, 99, TIMESTAMP '2026-03-01 12:00:00', 50.0),
  (5, 1,  TIMESTAMP '2026-06-01 00:00:00', 10.0)
AS txns(txn_id, customer_id, event_ts, amount)
""")

### The example data

- `customers` (8 rows) is one row per customer, with a region. Customers 7 and 8 (region East) have no orders.
- `orders` (14 rows) is one row per order, for customers 1 to 6. Most are `completed`; two are `cancelled` and one is
  `pending`.
- `segment_history` (3 rows) records each customer's segment over time. Customer 1 moved from `standard` to `premium`
  on 2026-06-01; customer 2 has always been `standard`.
- `txns` (5 rows) are transactions to look up against `segment_history`. Transaction 4 belongs to customer 99, who has
  no history, and transaction 5 falls exactly on customer 1's change date.

Timestamps are compared as strings or not collected at all in the asserts, because Python converts a collected
`TIMESTAMP` into the machine's local time zone.

In [ ]:
counts = spark.sql("""
SELECT (SELECT COUNT(*) FROM customers), (SELECT COUNT(*) FROM orders),
       (SELECT COUNT(*) FROM segment_history), (SELECT COUNT(*) FROM txns)
""").collect()
assert counts == [(8, 14, 3, 5)]

## 1. Start with the grain

The **grain** of a table is what one row represents. Write it as one sentence: "one row per order", "one row per
customer per day", "one row per change of segment".

The grain decides the table's **key**: the column or columns that are unique in every row. "One row per order" means
`order_id` is unique. "One row per customer per segment version" means `(customer_id, effective_from)` is unique.
Every join must agree with the grain, or the row count changes and sums go wrong.

A grain claim is testable. If the key is unique, `COUNT(*)` equals the count of distinct keys.

In [ ]:
q = """
SELECT
  (SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM orders)       AS dup_order_keys,
  (SELECT COUNT(*) - COUNT(DISTINCT customer_id) FROM customers) AS dup_customer_keys,
  (SELECT COUNT(*) - COUNT(DISTINCT customer_id, effective_from) FROM segment_history) AS dup_version_keys
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(0, 0, 0)]

### What a mixed grain does

`orders` is one row per order. `segment_history` is one row per customer per version. Joining them on `customer_id`
alone mixes the two grains: customer 1 has two versions, so each of its 3 orders appears twice.

Worked by hand: customer 1's orders total 100 + 200 + 150 = 450. After the join every order is doubled, so the same
`SUM` reports 900. Nothing errors; the number is just wrong. This is called **fan-out**.

In [ ]:
q = """
SELECT
  (SELECT SUM(amount) FROM orders WHERE customer_id = 1) AS true_total,
  (SELECT SUM(o.amount)
     FROM orders o JOIN segment_history s ON s.customer_id = o.customer_id
    WHERE o.customer_id = 1)                            AS fanned_out_total
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(450, 900)]

### Where you'll use it

- **Every fact table design:** the grain sentence ("one row per order line") is the first thing a dbt model or a
  data-modeling review asks for. Most double-counting bugs come from a mixed grain.
- **Data tests:** a `unique` test on the key column is the grain sentence written as code. dbt's `unique` and
  `unique_combination_of_columns` tests do exactly the check above.

## 2. The star schema

A **fact** table records events or measurements at a declared grain: each row holds numbers to add up (**measures**,
such as `amount`) and keys that point at the things involved (**foreign keys**, such as `customer_id`). A
**dimension** table describes one of those things, with one row per key and descriptive columns (`name`, `region`).

A **star schema** is one fact table with its dimension tables around it, each joined on its key. `orders` is a fact
table at the grain "one row per order". `customers` is a dimension keyed by `customer_id`.

Joining a fact table to a dimension should only add columns. It must not change the fact's row count. That holds only
when:

- the dimension key is unique, so no fact row matches twice, and
- every fact key exists in the dimension, so an inner join drops no fact rows (a fact row with no match is an
  **orphan**).

Check both. `orders` has 14 rows, and the join returns 14.

In [ ]:
q = """
SELECT
  (SELECT COUNT(*) - COUNT(DISTINCT customer_id) FROM customers)  AS duplicate_keys,
  (SELECT COUNT(*) FROM orders o
     LEFT JOIN customers c ON c.customer_id = o.customer_id
    WHERE c.customer_id IS NULL)                                  AS orphan_orders,
  (SELECT COUNT(*) FROM orders o
     JOIN customers c ON c.customer_id = o.customer_id)           AS fact_rows_after_join
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(0, 0, 14)]

Now break the rule. Suppose the dimension gains a second row for customer 1 (for example, a bad load). Each of
customer 1's 3 orders matches twice, so the join returns 14 + 3 = 17 rows, and every measure summed over the join is
inflated.

In [ ]:
q = """
WITH customers_bad AS (
  SELECT customer_id, region FROM customers
  UNION ALL
  SELECT 1, 'East'
)
SELECT COUNT(*) AS fact_rows_after_join
FROM orders o
JOIN customers_bad c ON c.customer_id = o.customer_id
"""
assert spark.sql(q).collect() == [(17,)]

### Where you'll use it

- **BI models:** Looker, Power BI and Tableau models are star schemas: a fact table in the middle, dimensions around it.
- **Pipeline tests:** "the joined row count equals the fact row count" is a cheap check to run after every load.

## 3. Slowly changing dimensions (Type 2)

A **slowly changing dimension** (SCD) is a dimension whose attributes change now and then, such as a customer's
segment or address. There are two common ways to store the change:

- **Type 1** overwrites the old value. The table always shows the current value, and history is lost.
- **Type 2** adds a new row per version, with `effective_from` and `effective_to`. History is kept.

The Type 2 interval is **half-open**: `[effective_from, effective_to)`. It includes the start and excludes the end, so
consecutive versions meet without overlapping. A NULL end means the version is still current.

To find the version that applied to each transaction, join on the customer and on a **range**: the transaction time
must be at or after the start and before the end. Worked by hand:

| txn | customer | event_ts | matching version |
|---|---|---|---|
| 1 | 1 | 2026-01-15 12:00 | standard (before 06-01) |
| 2 | 1 | 2026-07-01 12:00 | premium (from 06-01, open-ended) |
| 3 | 2 | 2026-03-01 12:00 | standard |
| 4 | 99 | 2026-03-01 12:00 | none: no history for customer 99 |
| 5 | 1 | 2026-06-01 00:00 | premium: the old version ends *before* 06-01 00:00 |

The `LEFT JOIN` keeps transaction 4 with a NULL segment.

In [ ]:
q = """
SELECT t.txn_id, t.customer_id, s.segment
FROM txns t
LEFT JOIN segment_history s
  ON s.customer_id = t.customer_id
 AND t.event_ts >= s.effective_from
 AND (s.effective_to IS NULL OR t.event_ts < s.effective_to)
ORDER BY t.txn_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [
    (1, 1, "standard"), (2, 1, "premium"), (3, 2, "standard"), (4, 99, None), (5, 1, "premium"),
]

### What Type 1 would have said

A Type 1 table keeps only the current row. Joining to it labels transaction 1 as `premium`, although customer 1 was
`standard` in January. Type 1 answers "what is the segment now?"; only Type 2 answers "what was it then?".

In [ ]:
q = """
SELECT t.txn_id, s.segment
FROM txns t
JOIN segment_history s
  ON s.customer_id = t.customer_id AND s.effective_to IS NULL
WHERE t.txn_id = 1
"""
assert spark.sql(q).collect() == [(1, "premium")]

### Why the end must be exclusive

If the end is inclusive (`t.event_ts <= s.effective_to`), transaction 5 at exactly 2026-06-01 00:00 matches both the
old version (which ends then) and the new one (which starts then). The join returns 6 rows for 5 transactions.

In [ ]:
q = """
SELECT COUNT(*) AS joined_rows
FROM txns t
LEFT JOIN segment_history s
  ON s.customer_id = t.customer_id
 AND t.event_ts >= s.effective_from
 AND (s.effective_to IS NULL OR t.event_ts <= s.effective_to)
"""
assert spark.sql(q).collect() == [(6,)]

### Testing that versions do not overlap

Before you trust a Type 2 table, check that no two versions of the same key overlap. An overlap makes the range join
return two rows for one transaction.

Two half-open intervals `[a_start, a_end)` and `[b_start, b_end)` overlap exactly when `a_start < b_end` and
`b_start < a_end`. A NULL end means "no end yet", so replace it with a far-future value. Number the versions per
customer and compare only pairs with `a.v < b.v`, so a row is never compared with itself and each pair is counted once.

The check below runs on the real table (0 overlaps) and on a bad copy where customer 2 gained a `premium` version from
2026-03-01 while the open `standard` version was never closed (1 overlap).

In [ ]:
OVERLAPS = """
WITH v AS (
  SELECT customer_id, effective_from,
         COALESCE(effective_to, TIMESTAMP '9999-12-31 00:00:00') AS effective_end,
         ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY effective_from, segment) AS v
  FROM {table}
)
SELECT COUNT(*) AS overlapping_pairs
FROM v a
JOIN v b
  ON a.customer_id = b.customer_id
 AND a.v < b.v
 AND a.effective_from < b.effective_end
 AND b.effective_from < a.effective_end
"""

spark.sql("""
CREATE OR REPLACE TEMP VIEW segment_history_bad AS
SELECT * FROM segment_history
UNION ALL
SELECT 2, 'premium', TIMESTAMP '2026-03-01 00:00:00', CAST(NULL AS TIMESTAMP)
""")

spark.sql(OVERLAPS.format(table="segment_history")).show()
assert spark.sql(OVERLAPS.format(table="segment_history")).collect() == [(0,)]
assert spark.sql(OVERLAPS.format(table="segment_history_bad")).collect() == [(1,)]

### Where you'll use it

- **History questions:** "what was the customer's region when they ordered?" needs a Type 2 dimension, with valid-from
  and valid-to dates.
- **dbt snapshots:** `dbt snapshot` builds Type 2 tables with `dbt_valid_from` and `dbt_valid_to` columns, using the
  same half-open convention.
- **Data quality checks:** the overlap test above, plus "at most one open version per key", runs after every load.

## 4. Reading a query plan

A **physical plan** is the tree of operators Spark will run. `EXPLAIN FORMATTED` prints it. Read it from the bottom
(the table scans) up to the top (the final result).

Two things matter most:

- **`Exchange`** is a **shuffle**: rows are redistributed across the cluster by a key, which means writing them out and
  sending them over the network. Shuffles are usually the most expensive step.
- **The join strategy.** A **sort-merge join** shuffles *both* tables by the join key, sorts each side, and merges
  them. A **broadcast hash join** copies the small table to every executor and builds a hash table from it, so the
  large table is joined where it already sits, with no shuffle.

Spark broadcasts automatically when a table is smaller than `spark.sql.autoBroadcastJoinThreshold` (10 MB by default),
so these tiny tables would be broadcast anyway. To see both strategies, turn automatic broadcasting off for a moment,
then ask for a broadcast with the `BROADCAST` hint.

In [ ]:
QUERY = """
SELECT {hint} c.region, SUM(o.amount) AS completed_total
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
WHERE o.status = 'completed'
GROUP BY c.region
ORDER BY c.region
"""

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
try:
    merge_plan = spark.sql("EXPLAIN FORMATTED " + QUERY.format(hint="")).first()[0]
    broadcast_plan = spark.sql("EXPLAIN FORMATTED " + QUERY.format(hint="/*+ BROADCAST(c) */")).first()[0]
    merge_rows = spark.sql(QUERY.format(hint="")).collect()
    broadcast_rows = spark.sql(QUERY.format(hint="/*+ BROADCAST(c) */")).collect()
finally:
    spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")

print(broadcast_plan.split("\n\n")[0])

The plan printed above has a `BroadcastHashJoin` fed by a `BroadcastExchange` of `customers`. The other plan has a
`SortMergeJoin`, with an `Exchange` that hash-partitions both sides by `customer_id`. The `Exchange` nodes left in
the broadcast plan belong to the `GROUP BY` and the `ORDER BY`, not to the join. The answer is identical either way;
only the amount of data moved changes.

In [ ]:
assert "SortMergeJoin" in merge_plan and "BroadcastHashJoin" not in merge_plan
assert "hashpartitioning(customer_id" in merge_plan
assert "BroadcastHashJoin" in broadcast_plan and "hashpartitioning(customer_id" not in broadcast_plan
assert merge_rows == broadcast_rows == [("North", 1150), ("South", 620), ("West", 1225)]

### Where you'll use it

- **Slow jobs and dashboards:** the plan shows the full table scan, the join that explodes rows, or the shuffle of a
  large table. Engineers read it before adding a partition, a hint or more memory.
- **Join tuning:** if a small dimension is being sort-merge joined, a broadcast hint or a higher threshold removes a
  shuffle of the large fact table.

## 5. Filters the engine can use

A **predicate** is a condition that is true or false for each row, such as `order_date >= DATE '2023-02-01'`. A
predicate is **sargable** (from "search argument able") when the engine can use it to skip data without reading it.

Skipping works because storage keeps summaries. Parquet files store the minimum and maximum of each column for each
block of rows; a B-tree index keeps the column sorted. Worked by hand: a block whose `order_date` ranges from
2023-01-02 to 2023-01-25 cannot contain a row with `order_date >= 2023-02-01`, so the engine skips it.

That reasoning needs a bare column compared with a constant. For `date_format(order_date, 'yyyy-MM') = '2023-02'` the
engine would have to know how the function maps a min/max range, and in general it does not. So it reads every block
and applies the function row by row.

To see this in Spark, write `orders` to Parquet and look at `PushedFilters` in the plan: the predicates handed to the
Parquet reader for skipping. The range form is pushed down. The function form is not; only `IsNotNull` is. Both
queries still count the same five February orders.

In [ ]:
import tempfile

parquet_path = tempfile.mkdtemp() + "/orders"
spark.table("orders").coalesce(1).write.parquet(parquet_path)
spark.read.parquet(parquet_path).createOrReplaceTempView("orders_parquet")

RANGE_FILTER = "order_date >= DATE '2023-02-01' AND order_date < DATE '2023-03-01'"
FUNCTION_FILTER = "date_format(order_date, 'yyyy-MM') = '2023-02'"


def pushed_filters(where):
    plan = spark.sql(f"EXPLAIN FORMATTED SELECT COUNT(*) FROM orders_parquet WHERE {where}").first()[0]
    return [line.strip() for line in plan.splitlines() if "PushedFilters" in line]


print(pushed_filters(RANGE_FILTER))
print(pushed_filters(FUNCTION_FILTER))

assert "GreaterThanOrEqual(order_date" in pushed_filters(RANGE_FILTER)[0]
assert "order_date" in pushed_filters(FUNCTION_FILTER)[0]
assert "GreaterThanOrEqual" not in pushed_filters(FUNCTION_FILTER)[0]
for where in (RANGE_FILTER, FUNCTION_FILTER):
    assert spark.sql(f"SELECT COUNT(*) FROM orders_parquet WHERE {where}").collect() == [(5,)]

One exception: when a table is **partitioned** by a column (stored as one folder per value), Spark evaluates the
predicate against the folder values themselves, so it can prune partitions even through a function. File statistics,
Parquet row groups, B-tree indexes and most warehouse pruning still need the bare-column form. Write range predicates
on the raw column by habit.

### Where you'll use it

- **Partition and file pruning:** a range filter on `order_date` lets the engine skip files and row groups; wrapping
  the column in a function disables that skipping.
- **Indexed lookups in PostgreSQL or MySQL:** `WHERE lower(email) = ...` cannot use a plain index on `email`; rewrite
  the predicate or index the expression.
- **Code review:** rewriting `year(d) = 2023` as `d >= '2023-01-01' AND d < '2024-01-01'` is one of the most common
  performance fixes.

## 6. Nested values

Spark columns can hold **arrays** (an ordered list of values) and **structs** (a fixed set of named fields, like a
row inside a cell). They suit denormalised reporting tables and semi-structured event data.

- `collect_set(x)` gathers the distinct values of a group into an array. `collect_list(x)` keeps duplicates. Neither
  guarantees an order, so wrap the result in `sort_array` when the order matters.
- `named_struct('id', ..., 'amount', ...)` builds a struct; `.amount` reads a field.
- `explode(array)` does the reverse of `collect_set`: one row per element.

Customer 2 has orders with statuses `completed`, `cancelled`, `completed`: three values, two distinct.

In [ ]:
q = """
SELECT customer_id,
       sort_array(collect_set(status)) AS statuses,
       size(collect_list(status))      AS status_count
FROM orders
WHERE customer_id = 2
GROUP BY customer_id
"""
spark.sql(q).show(truncate=False)
assert spark.sql(q).collect() == [(2, ["cancelled", "completed"], 3)]

q = """
SELECT named_struct('id', order_id, 'amount', amount).amount AS amount
FROM orders
WHERE order_id = 1
"""
assert spark.sql(q).collect() == [(100,)]

`explode` turns customer 2's array back into rows, one per distinct status.

In [ ]:
q = """
WITH nested AS (
  SELECT customer_id, sort_array(collect_set(status)) AS statuses
  FROM orders
  WHERE customer_id = 2
  GROUP BY customer_id
)
SELECT customer_id, explode(statuses) AS status
FROM nested
ORDER BY status
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(2, "cancelled"), (2, "completed")]

### Where you'll use it

- **Event data:** nested JSON or struct columns in BigQuery and Snowflake are flattened with `explode` (or `UNNEST` and
  `FLATTEN`) in staging models.
- **Reporting tables:** one row per customer with an array of products bought avoids a second table for a simple list.

## Interview notes

- **Grain first.** Every interview modeling question starts with "what is one row?". Answer it before naming tables.
- **Fan-out.** If a join changes the fact's row count, sums are wrong. Check key uniqueness before and row counts after.
- **SCD types.** Type 1 overwrites; Type 2 keeps a row per version with half-open validity dates. Know how to join a
  fact to a Type 2 dimension and how to test for overlaps.
- **Partitioning and pruning.** Partition by the column most queries filter on, usually a date. Keep partitions
  large enough to be worth reading.
- **Indexes.** Spark has no user-created indexes; it prunes partitions and skips files using min/max statistics.
  Row-store databases such as PostgreSQL use B-tree indexes for point lookups; columnar warehouses rely on
  partitioning, clustering and min/max pruning. The concepts chapter covers indexes and covering indexes.
- **Transactions.** Engines that lack them need idempotent writes instead, such as `INSERT OVERWRITE` for a
  partition. The views and changing-data chapter explains the trade-off.
- **Optimise by measuring.** Read the plan, change one thing, and compare. The concepts chapter lists the order to
  check things in.

## Summary

- Write the grain first, make every key and join agree with it, and test the key for uniqueness.
- A fact-to-dimension join must keep the fact's row count: the dimension key is unique and there are no orphans.
- Model changing attributes as half-open Type 2 intervals, join on a range, and test that versions do not overlap.
- Read the plan before you tune. It shows which join was chosen and where the shuffles (`Exchange`) are.
- Compare bare columns with constants, so the engine can skip data.
- Arrays and structs hold nested values; `collect_set` builds them and `explode` flattens them.

## Practice questions

1. For each region, how many completed orders are there, and what is their total amount? Use a join to `customers`,
   sorted by region. Customers 7 and 8 have no orders, so East does not appear.
2. Which segment applied to transactions 2, 3 and 5? Show `txn_id` and `segment`, sorted by `txn_id`.
3. Which distinct statuses does customer 4 have, as one sorted array?
4. A colleague joins `orders` to `segment_history` on `customer_id` only. How many rows does the join return, and how
   many orders do customers 1 and 2 actually have?
5. Rewrite `WHERE year(order_date) = 2023 AND month(order_date) = 3` as a sargable filter on `orders_parquet`. How many
   orders match, and is the range pushed down to Parquet?

## Answers

### Question 1

Join the fact to the dimension, filter to completed orders, and group by region. North is customers 1 and 2:
3 + 2 orders and 450 + 700 = 1150.

In [ ]:
q = """
SELECT c.region, COUNT(*) AS completed_orders, SUM(o.amount) AS completed_total
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
WHERE o.status = 'completed'
GROUP BY c.region
ORDER BY c.region
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("North", 5, 1150), ("South", 3, 620), ("West", 3, 1225)]

### Question 2

Use the half-open range join from section 3. Transaction 5 is exactly on the boundary, so it gets the new version.

In [ ]:
q = """
SELECT t.txn_id, s.segment
FROM txns t
LEFT JOIN segment_history s
  ON s.customer_id = t.customer_id
 AND t.event_ts >= s.effective_from
 AND (s.effective_to IS NULL OR t.event_ts < s.effective_to)
WHERE t.txn_id IN (2, 3, 5)
ORDER BY t.txn_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(2, "premium"), (3, "standard"), (5, "premium")]

### Question 3

`collect_set` gathers the distinct statuses; `sort_array` fixes their order.

In [ ]:
q = """
SELECT customer_id, sort_array(collect_set(status)) AS statuses
FROM orders
WHERE customer_id = 4
GROUP BY customer_id
"""
spark.sql(q).show(truncate=False)
assert spark.sql(q).collect() == [(4, ["completed", "pending"])]

### Question 4

Customer 1 has 3 orders and 2 versions (3 x 2 = 6 rows); customer 2 has 3 orders and 1 version (3 rows). The join
returns 9 rows for 6 orders, because it mixes the order grain with the version grain.

In [ ]:
q = """
SELECT
  (SELECT COUNT(*) FROM orders o JOIN segment_history s ON s.customer_id = o.customer_id) AS joined_rows,
  (SELECT COUNT(*) FROM orders WHERE customer_id IN (1, 2))                               AS actual_orders
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(9, 6)]

### Question 5

March 2023 is the half-open range `[2023-03-01, 2023-04-01)` on the bare column. Five orders match, and the plan shows
the range in `PushedFilters`.

In [ ]:
SARGABLE_MARCH = "order_date >= DATE '2023-03-01' AND order_date < DATE '2023-04-01'"
assert spark.sql(f"SELECT COUNT(*) FROM orders_parquet WHERE {SARGABLE_MARCH}").collect() == [(5,)]
assert spark.sql(
    "SELECT COUNT(*) FROM orders_parquet WHERE year(order_date) = 2023 AND month(order_date) = 3"
).collect() == [(5,)]
assert "LessThan(order_date,2023-04-01)" in pushed_filters(SARGABLE_MARCH)[0]